In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, FunctionTransformer
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Generate Dataset & Split
np.random.seed(42)
n_samples = 1000

df = pd.DataFrame({
    'income': np.random.exponential(50000, n_samples) + 20000,
    'loan_amount': np.random.exponential(15000, n_samples) + 5000,
    'credit_score': np.random.randint(550, 850, n_samples),
    'employment_type': np.random.choice(['Salaried', 'Self-Employed', 'Business', 'Unemployed'], n_samples, p=[0.5, 0.3, 0.15, 0.05]),
    'education': np.random.choice(['High School', 'Graduate', 'Post-Graduate'], n_samples, p=[0.3, 0.5, 0.2]),
    'application_date': pd.date_range(start='2023-01-01', periods=n_samples, freq='h'),
    'default': np.random.choice([0, 1], n_samples, p=[0.8, 0.2])
})

df.loc[df.sample(50, random_state=42).index, 'income'] = np.nan
df.loc[df.sample(30, random_state=42).index, 'employment_type'] = np.nan

X = df.drop(columns=['default'])
y = df['default']

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full)

def engineer_features(data):
    df_f = data.copy()
    df_f['application_date'] = pd.to_datetime(df_f['application_date'])
    df_f['app_month'] = df_f['application_date'].dt.month
    df_f['app_dayofweek'] = df_f['application_date'].dt.day_of_week
    df_f['income_loan_ratio'] = df_f['income'] / (df_f['loan_amount'] + 1)
    df_f['log_income'] = np.log1p(df_f['income'].fillna(df_f['income'].median()))
    df_f = df_f.drop(columns=['application_date'])
    return df_f

numeric_features = ['income', 'loan_amount', 'credit_score', 'app_month', 'app_dayofweek', 'income_loan_ratio', 'log_income']
categorical_features = ['employment_type']
ordinal_features = ['education']
education_categories = [['High School', 'Graduate', 'Post-Graduate']]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
        ('ord', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ordinal', OrdinalEncoder(categories=education_categories))]), ordinal_features)
    ]
)

print("=== 1. 4 CLASSIFIERS TRAINING ===")
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(oob_score=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

trained_pipes = {}
for name, clf in classifiers.items():
    pipe = Pipeline(steps=[('fe', FunctionTransformer(engineer_features)), ('preprocessor', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    trained_pipes[name] = pipe
    acc = pipe.score(X_val, y_val)
    oob = f" (OOB: {pipe.named_steps['clf'].oob_score_:.2f})" if hasattr(pipe.named_steps['clf'], 'oob_score') else ""
    print(f"{name} Validation Score: {acc:.2f}{oob}")

print("\n=== 2. LOGISTIC REGRESSION PROBABILITIES (5 Rows) ===")
lr_pipe = trained_pipes['Logistic Regression']
probs = lr_pipe.predict_proba(X_val[:5])
preds = lr_pipe.predict(X_val[:5])
print("Probabilities [Class 0, Class 1]:\n", np.round(probs, 3))
print("Predictions:", preds)

print("\n=== 3. REGRESSION (Predicting Loan Amount) ===")
reg_features = ['income', 'credit_score']
X_reg = X_train[reg_features].fillna(X_train[reg_features].median())
y_reg = X_train['loan_amount'].fillna(X_train['loan_amount'].median())

lin_reg = LinearRegression().fit(X_reg, y_reg)
ridge_reg = Ridge(alpha=1.0).fit(X_reg, y_reg)
lasso_reg = Lasso(alpha=1.0).fit(X_reg, y_reg)

print("Linear Regression Coefficients:", lin_reg.coef_)
print("Ridge Regression Coefficients:", ridge_reg.coef_)
print("Lasso Regression Coefficients:", lasso_reg.coef_)
print("Lasso Zero Coefficients Count:", np.sum(lasso_reg.coef_ == 0))

print("\n=== 4. K-MEANS CLUSTERING ===")
kmeans_data = X_train[['income', 'loan_amount']].fillna(0)
for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(kmeans_data)
    sil = silhouette_score(kmeans_data, km.labels_)
    print(f"K={k} -> Silhouette Score: {sil:.3f}")

=== 1. 4 CLASSIFIERS TRAINING ===
Logistic Regression Validation Score: 0.81
Decision Tree Validation Score: 0.62
Random Forest Validation Score: 0.80 (OOB: 0.80)
Gradient Boosting Validation Score: 0.79

=== 2. LOGISTIC REGRESSION PROBABILITIES (5 Rows) ===
Probabilities [Class 0, Class 1]:
 [[0.83  0.17 ]
 [0.801 0.199]
 [0.868 0.132]
 [0.847 0.153]
 [0.839 0.161]]
Predictions: [0 0 0 0 0]

=== 3. REGRESSION (Predicting Loan Amount) ===
Linear Regression Coefficients: [ 4.28051526e-03 -4.63136866e+00]
Ridge Regression Coefficients: [ 4.28051540e-03 -4.63136757e+00]
Lasso Regression Coefficients: [ 4.28053379e-03 -4.63122753e+00]
Lasso Zero Coefficients Count: 0

=== 4. K-MEANS CLUSTERING ===
K=2 -> Silhouette Score: 0.593
K=3 -> Silhouette Score: 0.465
K=4 -> Silhouette Score: 0.417
K=5 -> Silhouette Score: 0.446
